## Full fine-tuning training loop

In [1]:
import os
import torch
from torch.nn import functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
import math


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

### Local test

In [ ]:
from ShardDataLoader import create_dataloader
DATA_DIR = os.path.join(os.path.dirname("__file__"),"..","tinystories")
DATA_DIR = os.path.join(os.getcwd(),"..","tinystories")
train_dataloader = create_dataloader(shard_dir=DATA_DIR, split="train", batch_size=2,block_size=8, shuffle=True)
x,y = next(iter(train_dataloader))
print(x.shape, y.shape)

# Back to colab

In [3]:
from google.colab import drive
drive.mount("/content/drive")
ROOT_DIR = "/content/drive/MyDrive/Colab_Notebooks"
DATA_DIR = os.path.join(ROOT_DIR, "tinystories")
CKPT_DIR = os.path.join(ROOT_DIR, "finetune_log")

Mounted at /content/drive


#### DataLoader classes

In [4]:
# import os
# import multiprocessing as mp
# import numpy as np
# from transformers import GPT2TokenizerFast
# from datasets import load_dataset 
from tqdm import tqdm 
import time
import torch
import random
from torch.utils.data import IterableDataset, DataLoader

class ShardDataset(IterableDataset):
    """
    Iterable dataset from tokenized shards.
    Each shard is npy array of 1e6 token ids.
    Each example is
        x = tokens[i: i + block_size]
        y = tokens[i + 1: i + block_size + 1]
    """
    def __init__(
        self, 
        shard_dir,
        split = "train", 
        block_size = 1024, 
        shuffle = True, 
        seed = 1337,
        infinite = False
    ):
        super().__init__()

        self.shard_dir = shard_dir
        self.split = split
        self.block_size = block_size
        self.shuffle = shuffle
        self.seed = seed
        self.infinite = infinite
        shard_file_name = sorted([file for file in os.listdir(self.shard_dir) if ".npy" in file])[:3]
        
        if self.split == "train":
            # files except for _000000.npy
            self.shard_paths = [os.path.join(self.shard_dir,s) for s in shard_file_name if "_000000.npy" not in s]
        else: 
            # file _000000.npy is validation set
            self.shard_paths = [os.path.join(self.shard_dir,s) for s in shard_file_name if "_000000.npy" in s]
        if len(self.shard_paths) == 0:
            raise FileNotFoundError(f"No *.npy files found in {self.shard_dir}")
        _tokens = np.load(self.shard_paths[0],mmap_mode='r')
        seq_per_shard = _tokens.shape[0] // self.block_size # number of sequences per shard
        self._total_length = seq_per_shard * len(self.shard_paths)
        # print(_tokens.shape[0], shard_length, self._total_length)
    
    def __len__(self):
        return self._total_length
         

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is None:
            worker_id = 0
            num_workers = 1
        else:
            worker_id = worker_info.id
            num_workers = worker_info.num_workers

        rng = random.Random(self.seed + worker_id)
        shard_paths = self.shard_paths[worker_id::num_workers]
        if len(shard_paths) == 0:
            raise RuntimeError(
                f"Worker {worker_id} got no shards. "
                f"num_workers={num_workers}, num_shards={len(self.shard_paths)}"
            )
        while True:
            if self.shuffle:
                rng.shuffle(shard_paths)
            # for p in shard_paths:
            #     print(p)
            for shard_path in shard_paths:
                tokens = np.load(shard_path,mmap_mode='r').astype(np.int32)
                if len(tokens) <= self.block_size + 1:
                    continue
                max_start = len(tokens) - (self.block_size + 1)
                start_indices = list(range(0, max_start,self.block_size))
                if self.shuffle: 
                    rng.shuffle(start_indices)
                for start in start_indices:
                    chunk = tokens[start: start + self.block_size + 1]
                    x = torch.tensor(chunk[:-1], dtype=torch.long)
                    y = torch.tensor(chunk[1:], dtype=torch.long)
                    yield x, y
            if not self.infinite: # not infinite dataloader, for epoch control
                break

def create_dataloader(
    shard_dir,
    split = "train", 
    batch_size = 32, 
    block_size = 1024,
    shuffle = True, 
    seed = 1337
):
    shard_dataset = ShardDataset(
        shard_dir = shard_dir, 
        split = split, 
        block_size = block_size , 
        shuffle = shuffle, 
        seed = seed
    )
    shard_dataloader = DataLoader(
        shard_dataset,
        batch_size = batch_size
    )
    return shard_dataloader


## DataLoaders

In [5]:
BATCH_SIZE = 32 #// 16
BLOCK_SIZE = 1024 #// 16
BATCH_SIZE, BLOCK_SIZE


(32, 1024)

In [6]:
train_dataloader = create_dataloader(
    shard_dir=DATA_DIR, 
    split="train", 
    batch_size=BATCH_SIZE,
    block_size=BLOCK_SIZE, 
    shuffle=True
)
valid_dataloader = create_dataloader(
    shard_dir=DATA_DIR, 
    split="test", 
    batch_size=BATCH_SIZE,
    block_size=BLOCK_SIZE, 
    shuffle=False
)

In [7]:
1e6/BLOCK_SIZE/BATCH_SIZE, len(train_dataloader),len(valid_dataloader)

(30.517578125, 61, 31)

In [8]:
train_dataloader.batch_size, getattr(train_dataloader.dataset, "block_size", None)

(32, 1024)

## Base model from Hugging Fase

In [ ]:
from huggingface_hub import login
hf_token = "" # removed in public repo 
login(hf_token, add_to_git_credential=True) 

In [10]:
base_model_name = "littleBallOfFur/baby-gpt-base"
tokenizer = AutoTokenizer.from_pretrained(base_model_name,trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(base_model_name,trust_remote_code=True).to(device)
base_model = AutoModelForCausalLM.from_pretrained(base_model_name,trust_remote_code=True).to(device)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/584 [00:00<?, ?B/s]

babygpt_model.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/littleBallOfFur/baby-gpt-base:
- babygpt_model.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/652M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

In [11]:
def reset_model():
    model = AutoModelForCausalLM.from_pretrained(base_model_name,trust_remote_code=True).to(device)
    return model

In [ ]:
import wandb
wb_token = "" # removed in public repo
wandb.login(key=wb_token)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: greygreygreat (Overfit&Coupled) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [19]:
from huggingface_hub import HfApi, create_repo, hf_hub_download
from safetensors.torch import save, load_file
import io
from datetime import datetime

PROJECT_NAME = "babygpt-raw-finetune"
HUB_SUFFIX = "" # save to hub under this
HUB_MODEL_NAME = f"littleBallOfFur/{PROJECT_NAME}{HUB_SUFFIX}"

def log_local(log_dir, file_name, content):
    os.makedirs(log_dir, exist_ok=True)
    log_file = os.path.join(log_dir,file_name)
    log_txt = ",".join([f"{k} {v}" for k, v in content.items()])
    with open(log_file, "a") as f:
        f.write(f"{log_txt}\n")
def log_wandb(content):
    step = content.pop('step')
    wandb.log(content,step=step)
    # print("wandb log: ",content,step)

def log(local_wandb, *args, **kwargs):
    if local_wandb == "local":
        log_local( *args, **kwargs)
    if local_wandb == "wandb":
        log_wandb( *args, **kwargs)


def save_checkpoint(hf_local,*args, **kwargs):
    if hf_local == "local":
        save_checkpoint_local(*args, **kwargs)
    if hf_local == "hf":
        save_checkpoint_hf(*args, **kwargs)
        
def save_checkpoint_hf(model, repo_id, step, path_in_repo="model.safetensors",private=True):
    create_repo(
        repo_id=repo_id,
        repo_type="model",
        private=private,
        exist_ok=True
    )
    # Move tensors to CPU before serialization.
    # Also detach to avoid autograd references.
    state_dict = {
        k: v.detach().cpu()
        for k, v in model.state_dict().items()
    }

    # Serialize to safetensors bytes in memory.
    safetensors_bytes = save(state_dict)

    buffer = io.BytesIO(safetensors_bytes)

    api = HfApi()
    api.upload_file(
        path_or_fileobj=buffer,
        path_in_repo=path_in_repo,
        repo_id=repo_id,
        repo_type="model",
        commit_message=f"Upload {path_in_repo} at step {step}",
    )
    print(f"saved checkpoint to: {repo_id}\n")

def save_checkpoint_local(model, ckpt_dir, model_name, step, val_loss, optimizer=None,config=None):
    os.makedirs(ckpt_dir, exist_ok=True)
    if ".pt" not in model_name:
        model_name = model_name + ".pt"
    checkpoint = {
        "model": model.state_dict(),
        "step": step,
        "val_loss": val_loss
    }
    if optimizer is not None:
        checkpoint["optimizer"] = optimizer.state_dict()
    if config is not None:
        checkpoint["config"] = config
    checkpoint_path = os.path.join(ckpt_dir,model_name)
    torch.save(checkpoint, checkpoint_path)
    print(f"saved checkpoint to: {checkpoint_path}\n")

def load_checkpoint(ckpt_dir, model_name, base_model=None):
    if ".pt" not in model_name:
        model_name = model_name + ".pt"
    checkpoint_path = os.path.join(ckpt_dir,model_name)
    checkpoint = torch.load(checkpoint_path, map_location='cpu',weights_only=False)
    state_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint
    if base_model:
        missing, unexpected = base_model.load_state_dict(state_dict)
        print("loaded checkpoint from: ", checkpoint_path)
        print(f"missing {missing} | unexpected {unexpected}")
        return base_model
    else:
        print("returning checkpoint state_dict from: ", checkpoint_path)
        return state_dict

def load_checkpoint_hf(base_model, repo_id, file_name = "model.safetensors"):
    state_dict = load_file(hf_hub_download(
        repo_id=repo_id,
        filename=file_name,
    ))
    missing, unexpected = base_model.load_state_dict(state_dict)
    print("loaded checkpoint from: ", repo_id)
    print(f"missing {missing} | unexpected {unexpected}")
    return base_model


In [ ]:
torch.set_float32_matmul_precision('high')

@torch.no_grad()
def generate_sample(
    model, 
    device, 
    tokenizer, 
    prompt = "Once upon a time",
    max_new_tokens = 50,
    seed = 1337
):
    model.eval()
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    input_tokens = tokenizer.encode(prompt, return_tensors='pt').to(device)
    output_tokens = model.generate(input_tokens, 
                            max_new_tokens=max_new_tokens,
                            do_sample=True,
                            top_k = 50)
    
    output_text = tokenizer.decode(output_tokens[0])
    return output_text

def get_lr(step, max_steps, warmup_steps, max_lr, min_lr):
    if step <= warmup_steps:
        lr = max_lr * step/warmup_steps
    elif step > max_steps:
        lr = min_lr
    else:
        pi_fraction = (step - warmup_steps) / (max_steps - warmup_steps)
        ratio = 0.5 * (1 + math.cos(math.pi * pi_fraction))
        lr = min_lr + ratio * (max_lr - min_lr)
    return lr

@torch.no_grad()
def eval_loop(model, valid_dataloader, device, eval_batches = 50):
    model.eval()
    batch = 0
    losses = []
    for (x,y) in valid_dataloader:
        batch += 1
        if batch >= eval_batches:
            break
        x, y = x.to(device), y.to(device)
        with torch.autocast( device_type=device.type,dtype=torch.bfloat16,enabled=(device.type == "cuda")):
            logits = model(x).logits
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
        losses.append(loss.item())
    return np.array(losses).mean()

def training_loop(
    model,
    train_dataloader,
    valid_dataloader,
    device,
    num_epochs = 1,
    max_steps = 20,
    grad_accum_steps = 2, # num of batches per step
    max_lr = 3e-5,
    min_lr = 3e-6,
    warmup_steps = 5,
    weight_decay = 0.1,
    betas = (0.9, 0.95),
    grad_clip = 1.0,
    eval_every = 10,
    eval_batches = 5,
    log_every = 5,

    ckpt_dir = "checkpoints",
    local_ckpt = False,
    hf_repo_id = None,

    local_log = True,
    use_wandb = False,
    wandb_project = "dummy_project",
):
    model.train()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=max_lr,
        betas=betas,
        weight_decay=weight_decay,
    )
    step = 0 # optimizer steps
    steps_per_epoch = math.ceil(len(train_dataloader) / grad_accum_steps)
    if max_steps > 0: # early stop at max_steps
        max_steps = min(max_steps, num_epochs * steps_per_epoch)
    else:
        max_steps = num_epochs * steps_per_epoch
    print(f"{steps_per_epoch} steps per epoch , max steps {max_steps}, {len(train_dataloader)} batches per epoch ")
    
    if warmup_steps<1:
        warmup_steps = int(max_steps * warmup_steps)
    
    # wandb setup
    config = {
        "num_epochs": num_epochs,
        "steps_per_epoch": steps_per_epoch,
        "max_steps": max_steps,
        "grad_accum_steps": grad_accum_steps,
        "max_lr": max_lr,
        "min_lr": min_lr,
        "warmup_steps": warmup_steps,
        "weight_decay": weight_decay,
        "betas": betas,
        "grad_clip": grad_clip,
        "batch_size": train_dataloader.batch_size,
        "seq_len": getattr(train_dataloader.dataset, "block_size", None),
    }
    if use_wandb:
        RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
        wandb.init(project=wandb_project, name=RUN_NAME, config=config)

    # Training loop
    for epoch in range(num_epochs):
        print(f"\n ====== epoch {epoch+1}/{num_epochs} ======")

        loss_accum = 0
        micro_step = 0 # for grad_accum
        optimizer.zero_grad()
        t0 = time.time()

        for x,y in train_dataloader:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled = (device.type=="cuda")):
                logits = model(x).logits
            # BabyGPT implements shifted loss, so manually calculate here
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
            loss = loss / grad_accum_steps
            loss_accum += loss.item()
            loss.backward()
            micro_step += 1 # also batch number

            should_update_grad = micro_step % grad_accum_steps == 0
            is_last_batch = micro_step == len(train_dataloader) 
            is_last_step = step == max_steps - 1
            # TODO: last step did not fully accum loss, results in incorrect low loss and grad
            if should_update_grad or is_last_step:
                lr = get_lr(step, max_steps, warmup_steps, max_lr, min_lr)
                for param_group in optimizer.param_groups:
                    param_group["lr"] = lr
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(),grad_clip)
                optimizer.step()
                optimizer.zero_grad()
                step += 1    
                if step % log_every == 0 or is_last_step:
                    dt = (time.time() - t0)/log_every
                    t0 = time.time()
                    print(
                        f"step {step:6d}/{max_steps} | "
                        f"batch {micro_step:6d}/{len(train_dataloader)} | "
                        f"loss {loss_accum:.4f} | "
                        f"lr {lr:.2e} | "
                        f"dt {dt:.2f}s"
                    )
                    log_content = {
                        "step":step,
                        "train/loss":loss_accum,
                        "train/lr":lr,
                        "train/grad_norm":grad_norm.item(),
                        # "train/epoch":epoch+1,
                        "train/tokens_per_sec": BATCH_SIZE * BLOCK_SIZE * grad_accum_steps/dt
                    }
                    if local_log:
                        log("local",ckpt_dir,"log.txt",log_content)
                    if use_wandb:
                        log("wandb",log_content)
                
                if step % eval_every == 0 or is_last_step:
                    valid_loss = eval_loop(model, valid_dataloader, device, eval_batches = eval_batches)
                    train_loss = eval_loop(model, train_dataloader, device, eval_batches = eval_batches)
                    sample_output = generate_sample(model,device,tokenizer)
                    model.train()
                    print(f"\n----- eval at step {step:6d}/{max_steps} -----"
                          f"\ntraining loss {train_loss:.4f} | "
                          f"validation loss {valid_loss:.4f}"
                          f"\nsample output: {sample_output}\n---------------")
                    
                    log_content = {
                        "step":step,
                        "eval/val_loss":valid_loss,
                        "eval/val_ppl":np.exp(valid_loss),
                        # "eval/epoch":epoch+1
                        "eval/train_loss":train_loss
                    }
                    if local_log:
                        log("local",ckpt_dir,"log.txt",log_content)
                    if use_wandb:
                        log("wandb",log_content)
                    
                    if hf_repo_id:
                        save_checkpoint("hf", model, hf_repo_id,step)
                    if local_ckpt:
                        model_name = f"step_{step:06d}"
                        save_checkpoint("local",model,ckpt_dir, model_name,step,valid_loss)
                    
                loss_accum = 0
                
                if step >= max_steps: # could early stop
                    if use_wandb: wandb.finish()
                    return model
    if use_wandb: wandb.finish()
    return model
            



In [74]:
# for epoch in range(2):
#     print ("epoch ",epoch)
#     count_batches = 0
#     print(len(train_dataloader))
#     iter_train_dataloader = iter(train_dataloader)
#     for i, (x,y) in enumerate(train_dataloader):
#         count_batches +=1
#         print(x[0,:4], y[0,:4],i)
#     print(count_batches,i)

In [76]:
# step = 6
# valid_loss = 99
# ckpt_dir = CKPT_DIR
# model_name = f"step_{step:06d}"
# save_checkpoint('local',model,ckpt_dir, model_name,step,valid_loss)
# save_checkpoint('hf',model,repo_id = HUB_MODEL_NAME, step = step )

In [22]:

trained_model = training_loop(
    model,
    train_dataloader,
    valid_dataloader,
    device, 
    grad_accum_steps= 1,
    num_epochs=1,
    max_steps=-1,
    log_every=2, 
    eval_every=5,
    eval_batches = 50,
    ckpt_dir = CKPT_DIR,
    local_ckpt = False,
    hf_repo_id = HUB_MODEL_NAME, # None if don't save to HF,
    local_log=True,
    use_wandb = True,
    wandb_project= PROJECT_NAME
)

61 steps per epoch , max steps 61, 61 batches per epoch 



 ====== epoch 1/1 ======
step      2/61 | batch      2/61 | loss 1.6094 | lr 6.00e-06 | dt 0.11s
step      4/61 | batch      4/61 | loss 1.5469 | lr 1.80e-05 | dt 0.15s


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



----- eval at step      5/61 -----
training loss 1.6051 | validation loss 1.6895
sample output: Once upon a time, in a small house, there was a little girl named Lily. She had a beautiful pink ball. Lily liked to spin around and around under the bed, enjoying the sunshine every day. One sunny day, the ball went for a visit from her
---------------


Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



----- eval at step     61/61 -----
training loss 1.5214 | validation loss 1.6288
sample output: Once upon a time, in a small house, there was a little fairy. The fairy was very curious about the flowers.
One day, a little girl named Sue was playing with a big, friendly lion. The lion was very fast. Sue wanted to see what
---------------


Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


saved checkpoint to: littleBallOfFur/babygpt-raw-finetune



eval/train_loss,█▇▆▅▄▃▃▂▂▁▁▁▁
eval/val_loss,█▇▆▅▄▄▃▂▂▁▁▁▁
eval/val_ppl,█▇▆▅▄▃▃▂▂▁▁▁▁
train/grad_norm,▁▂▆█▄▄▆▃▃▆▄▃▂▅▅▃▃▄▃▃▃▃▃▃▃▂▃▃▂▂▁
train/loss,▆▃▁▃▄▁▄▄▃▅▄▄▆▄▆▅▆▆▇▅▆▆▆▅▅▆▆▆██▅
train/lr,▂▅█████▇▇▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁
train/tokens_per_sec,█▆▁▆▆▁▆▁▆▆▁▆▁▆▆▁▆▁▆▆▁▆▁▆▆▁▆▁▆▆▁
eval/train_loss,1.52136
eval/val_loss,1.62878
eval/val_ppl,5.09765
train/grad_norm,0.93035


In [32]:
ckpt_model = reset_model()
ckpt_model = load_checkpoint(CKPT_DIR, "step_000006",ckpt_model)
# print(generate_sample(ckpt_model,device,tokenizer))

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

loaded checkpoint from:  /content/drive/MyDrive/Colab_Notebooks/finetune_log/step_000006.pt
missing [] | unexpected []


In [33]:
hf_model = reset_model()
hf_model = load_checkpoint_hf(hf_model, HUB_MODEL_NAME)
# print(generate_sample(hf_model,device,tokenizer))

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/652M [00:00<?, ?B/s]

loaded checkpoint from:  littleBallOfFur/babygpt-raw-finetune
missing [] | unexpected []


In [34]:
print(generate_sample(base_model,device,tokenizer))
print("="*20)
print(generate_sample(model,device,tokenizer))
print("="*20)
print(generate_sample(ckpt_model,device,tokenizer))
print("="*20)
print(generate_sample(hf_model,device,tokenizer))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time, we were talking about the way we’d be building a church. This was about the way we were going to lay out the foundation. And so this is a very familiar phrase in this church building.
So as you can see they


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time, there was a lovely garden in the middle of a large park. I lived in a tiny city, surrounded by trees and blue skies. But there was an annoying problem: There was no garden to share this beautiful space with. The kids had fun


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time, there was a lovely garden in the middle of a large park. I lived in a tiny city, surrounded by trees and blue skies. But there was an annoying problem: There was no garden to share this beautiful space with. The kids had fun
Once upon a time, there was a lovely garden in the middle of a large park. I lived in a tiny city, surrounded by trees and blue skies. But there was an annoying problem: There was no garden to share this beautiful space with. The kids had fun
